In [1]:
%%writefile /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/requirements.txt
boto3

Writing /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/requirements.txt


In [2]:
%%writefile /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/app.py
import os
import sys
from datetime import datetime
from pathlib import Path

print("=======================================================")
print("🚀 [K8S BATCH] PROCESADOR DE TELEMETRÍA AUTOMOTRIZ V6")
print("=======================================================")

# Definimos rutas dentro del contenedor
BASE_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = BASE_DIR / "data" / "alerts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Lote simulado de entrada (Telemetría de la flota)
lote_vehiculos = [
    {"vin": "VIN-75101", "rpm": 2200, "temp": 92, "falla": "NONE"},
    {"vin": "VIN-75102", "rpm": 3200, "temp": 105, "falla": "P0300"},  # Misfire detectado
    {"vin": "VIN-75103", "rpm": 1800, "temp": 89, "falla": "NONE"},
    {"vin": "VIN-75104", "rpm": 2900, "temp": 101, "falla": "P0171"},  # Mezcla pobre detectada
]

print(f"📥 Leyendo lote de control... {len(lote_vehiculos)} registros encontrados.")

alertas_detectadas = []
for vehiculo in lote_vehiculos:
    if vehiculo["falla"] != "NONE":
        print(f"⚠️  Desviación Crítica en {vehiculo['vin']} -> Código: {vehiculo['falla']}")
        vehiculo["procesado_at"] = str(datetime.now())
        alertas_detectadas.append(vehiculo)

# Guardar reporte de salida
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
reporte_path = OUTPUT_DIR / f"criticos_{timestamp}.txt"

with open(reporte_path, "w", encoding="utf-8") as f:
    f.write(f"--- REPORTE DE ALERTAS FLOTA ({datetime.now()}) ---\n")
    for alerta in alertas_detectadas:
        f.write(f"Vehículo: {alerta['vin']} | RPM: {alerta['rpm']} | Temp: {alerta['temp']}°C | Error: {alerta['falla']}\n")

print(f"💾 Reporte de auditoría generado con éxito en: {reporte_path.name}")
print("=======================================================")
print("🏁 [ÉXITO] Lote procesado al 100%. Balance de infraestructura OK.")
print("Cerrando contenedor de forma limpia...")

# Clave para Kubernetes: Salida código 0 significa "Trabajo terminado con éxito"
sys.exit(0)

Writing /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/app.py


In [3]:
%%writefile /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Copiar dependencias e instalar
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiar el script de la aplicación
COPY app.py .

# Comando por defecto al arrancar el contenedor
CMD ["python", "app.py"]

Writing /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/Dockerfile


In [ ]:
!cd /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container && docker build -t batch-telemetria:v1 .

In [6]:
%%writefile /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/job.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: batch-telemetria-job
spec:
  template:
    spec:
      containers:
      - name: procesador-telemetria
        image: batch-telemetria:v1
        # IMPORTANTE: Como la imagen está en tu Docker local, le decimos a K8s que no intente buscarla en el internet (Docker Hub)
        imagePullPolicy: Never
      
      # Regla de oro en K8s para procesos Batch: Si el contenedor termina con éxito, no lo reinicies.
      restartPolicy: Never
  
  # Si por alguna razón el script falla (exit 1), K8s reintentará la tarea máximo 3 veces antes de darse por vencido.
  backoffLimit: 3

Writing /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/job.yaml


In [7]:
!kubectl apply -f /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/job.yaml

job.batch/batch-telemetria-job created


In [8]:
!kubectl get pods -l job-name=batch-telemetria-job

NAME                         READY   STATUS      RESTARTS   AGE
batch-telemetria-job-kpjbv   0/1     Completed   0          101s


In [9]:
!kubectl logs -l job-name=batch-telemetria-job

🚀 [K8S BATCH] PROCESADOR DE TELEMETRÍA AUTOMOTRIZ V6
📥 Leyendo lote de control... 4 registros encontrados.
⚠️  Desviación Crítica en VIN-75102 -> Código: P0300
⚠️  Desviación Crítica en VIN-75104 -> Código: P0171
💾 Reporte de auditoría generado con éxito en: criticos_20260704_233925.txt
🏁 [ÉXITO] Lote procesado al 100%. Balance de infraestructura OK.
Cerrando contenedor de forma limpia...


In [10]:
%%writefile app.py
import io
import os
from pathlib import Path
from datetime import datetime
import boto3
from botocore.config import Config
from azure.storage.blob import BlobServiceClient

# =========================================================
# CAPTURA DE VARIABLES DE ENTORNO (EL PUENTE CON DOCKER)
# =========================================================
SIMULATION_ENV = os.environ.get("SIMULACION", "True")
SIMULATION_MODE = SIMULATION_ENV.upper() == "TRUE"

AWS_BUCKET = os.environ.get("AWS_INPUT_BUCKET", "fleet-raw-data")
AWS_KEY = os.environ.get("AWS_INPUT_KEY", "raw_telemetry.csv")
AZURE_CONTAINER = os.environ.get("AZURE_OUTPUT_CONTAINER", "fleet-clean-alerts")

# Definición de Directorios Base de forma robusta
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
DATA_DIR = BASE_DIR / "data"
LOCAL_INPUT_ARCHIVE = DATA_DIR / "archive" / "input"
LOCAL_OUTPUT_ARCHIVE = DATA_DIR / "archive" / "output"

def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria...")
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\n"
        )
    else:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        s3_data = response["Body"].read().decode("utf-8")
    
    LOCAL_INPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_raw_file = LOCAL_INPUT_ARCHIVE / f"raw_s3_mirror_{timestamp_str}.csv"
    local_raw_file.write_text(s3_data, encoding="utf-8")
    print(f"💾 [Auditoría Local] Réplica cruda de entrada guardada en: {local_raw_file.relative_to(BASE_DIR)}")

    stream_buffer = io.StringIO(s3_data)
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def azure_bulk_writer(container_client, blob_name, processed_records, simulation_mode=False):
    if not processed_records:
        return
    
    headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    csv_buffer = headers
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    if simulation_mode:
        print("☁️  [Simulación] Modo local activo. Se saltó la carga real a Azure.")
    else:
        container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)
        print("☁️  [Cloud] Transmisión exitosa a Azure Blob Storage.")
    
    # CORRECCIÓN: Forzamos la creación del directorio dentro de /app/data
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_OUTPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    # Aseguramos que el archivo de texto de auditoría se guarde dentro del volumen mapeado
    local_clean_file = DATA_DIR / f"criticos_{timestamp_str}.txt"
    local_clean_file.write_text(csv_buffer, encoding="utf-8")
    print(f"💾 [Auditoría Local] Reporte filtrado guardado en con éxito en: {local_clean_file.relative_to(BASE_DIR)}")

if __name__ == '__main__':
    print("🚀 [K8S BATCH] PROCESADOR DE TELEMETRÍA AUTOMOTRIZ V6")
    print("=======================================================")
    
    AWS_BUCKET = os.getenv("AWS_INPUT_BUCKET", "dev-fleet-raw-data")
    AWS_KEY = os.getenv("AWS_INPUT_KEY", "daily_telemetry.csv")
    AZURE_CONTAINER = os.getenv("AZURE_OUTPUT_CONTAINER", "dev-fleet-alerts")
    
    SIMULACION = os.getenv("SIMULACION", "True").upper() == "TRUE"
    
    s3, azure_container_client = None, None
    
    if not SIMULACION:
        from moto import mock_aws
        mock_context = mock_aws()
        mock_context.start()
        
        s3 = boto3.client("s3", region_name="us-east-1")
        s3.create_bucket(Bucket=AWS_BUCKET)
        
        headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
        bulk_data = headers
        # Inyección de los registros simulando las desviaciones detectadas (P0300 y P0171)
        bulk_data += f"{datetime.now()},VIN-75101,2100,90,NONE\n"
        bulk_data += f"{datetime.now()},VIN-75102,3500,105,P0300\n"
        bulk_data += f"{datetime.now()},VIN-75103,1900,88,NONE\n"
        bulk_data += f"{datetime.now()},VIN-75104,2900,101,P0171\n"
            
        s3.put_object(Bucket=AWS_BUCKET, Key=AWS_KEY, Body=bulk_data.encode("utf-8"))
        
        class LocalMockAzureContainer:
            def upload_blob(self, name, data, overwrite=True):
                pass
        
        azure_container_client = LocalMockAzureContainer()
        
    else:
        # Caída en modo simulación integrado de 3 filas
        pass
    
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULACION)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            print(f"⚠️  Desviación Crítica en {raw_row['vehicle_id']} -> Código: {raw_row['fault_code']}")
            incidents.append(clean_row)
            
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents, simulation_mode=SIMULACION)
    
    print("=======================================================")
    print("🏁 [ÉXITO] Lote procesado al 100%. Balance de infraestructura OK.")
    print("Cerrando contenedor de forma limpia...")

Overwriting app.py


In [11]:
!docker compose up --build -d

no configuration file provided: not found


In [12]:
%%writefile docker-compose.yml
version: '3.8'

services:
  localstack:
    image: localstack/localstack:2.3.2
    container_name: localstack_simulador
    ports:
      - "127.0.0.1:4566:4566"
    environment:
      - SERVICES=s3,sqs,dynamodb
      - AWS_DEFAULT_REGION=us-east-1
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:4566/_localstack/health"]
      interval: 3s
      timeout: 3s
      retries: 5

  fleet-worker:
    build: .
    image: fleet-multicloud-worker:local
    container_name: fleet_batch_worker_instance
    environment:
      - SIMULACION=False
      - AWS_INPUT_BUCKET=dev-fleet-raw-data
      - AWS_INPUT_KEY=daily_telemetry.csv
      - AZURE_OUTPUT_CONTAINER=dev-fleet-alerts
      - AWS_ACCESS_KEY_ID=mock_key_para_pruebas_locales
      - AWS_SECRET_ACCESS_KEY=mock_secret_para_pruebas_locales
      - AWS_ENDPOINT_URL=http://localstack:4566
    volumes:
      - ./data:/app/data
    depends_on:
      localstack:
        condition: service_healthy

  scheduler:
    image: mcuadros/ofelia:latest
    container_name: local_cloud_scheduler
    volumes:
      - /var/run/docker.sock:/var/run/docker.sock:ro
    command: daemon --docker
    labels:
      ofelia.job-run.fleet-batch-trigger.schedule: "*/1 * * * *"
      ofelia.job-run.fleet-batch-trigger.container: "fleet_batch_worker_instance"

Writing docker-compose.yml


In [13]:
!ls -la docker-compose.yml Dockerfile app.py

-rw-r--r--  1 admin  staff   271 Jul  4 16:19 Dockerfile
-rw-r--r--  1 admin  staff  6400 Jul  4 16:45 app.py
-rw-r--r--  1 admin  staff  1310 Jul  4 16:47 docker-compose.yml


In [16]:
!docker compose up --build -d

[+] Building 0.0s (0/0)                                                         
[+] Building 0.0s (0/0)                                                         
[+] Building 0.0s (0/0)                                                         
[+] Building 0.0s (0/0)                                                         
[+] Building 0.0s (0/0)                                                         
[+] Building 0.0s (0/0)                                                         
[+] Building 0.0s (0/0)                                                         
[+] Building 0.1s (1/1)                                                         
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 32B                                        0.0s
[+] Building 0.3s (2/2)                                                         
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile:

In [17]:
!docker ps

CONTAINER ID   IMAGE                         COMMAND                  CREATED         STATUS                   PORTS                                               NAMES
2081636f954f   localstack/localstack:2.3.2   "docker-entrypoint.sh"   3 minutes ago   Up 3 minutes (healthy)   4510-4559/tcp, 5678/tcp, 127.0.0.1:4566->4566/tcp   localstack_simulador
1e1f7aa16cdb   mcuadros/ofelia:latest        "/usr/bin/ofelia dae…"   3 minutes ago   Up 3 minutes                                                                 local_cloud_scheduler
